# Simulation Benchmark Aggregate

Read-only aggregation for the method-split simulation benchmarks. This notebook validates and combines the four split output roots without running any decompositions or graph estimators.

Expected split roots:

- `outputs/simulation/core_1000_fastica/`
- `outputs/simulation/core_1000_infomax/`
- `outputs/simulation/core_1000_jade/`
- `outputs/simulation/core_1000_sobi/`

Outputs are written to `outputs/simulation/core_1000_aggregate/summary/` and `outputs/simulation/core_1000_aggregate/figures/`.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "ica_denoising").exists():
            return path
    raise RuntimeError("Could not find the ica-denoising repository root.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".cache" / "matplotlib"))
(REPO_ROOT / ".cache" / "matplotlib").mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib

if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

if "ipykernel" in sys.modules:
    try:
        from IPython import get_ipython

        ip = get_ipython()
        if ip is not None:
            ip.run_line_magic("matplotlib", "inline")
    except Exception:
        pass

from ica_denoising.simulation import load_config
from ica_denoising.simulation.dataset import default_scenarios
from ica_denoising.simulation.reporting import collect_metric_table
from ica_denoising.simulation.validate import validate_benchmark

sns.set_theme(style="whitegrid", context="talk")
SHOW_FIGURES = "ipykernel" in sys.modules
print(f"Repository root: {REPO_ROOT}")

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs" / "simulation.core.json"
BASE_BENCHMARK_VERSION = "core_1000"
METHOD_SPLITS = ("fastica", "infomax", "jade", "sobi")
BSS_METHODS = {"fastica", "infomax", "jade", "sobi"}

# Keep False for final manuscript aggregation. Set True only for progress/debug previews.
ALLOW_PARTIAL = False

MAIN_GRAPH_ESTIMATORS = ("cgc", "cgc_star", "pcmci", "jpcmciplus")
SUPPLEMENTARY_GRAPH_ESTIMATORS = ("var",)

cfg = load_config(CONFIG_PATH)
scenario_ids = [s.scenario_id for s in (list(cfg.scenarios) or list(default_scenarios()))]
seed_ids = list(cfg.seeds)
expected_replicates_per_method = len(scenario_ids) * len(seed_ids)

split_roots = {
    method: REPO_ROOT / cfg.run.output_dir / f"{BASE_BENCHMARK_VERSION}_{method}"
    for method in METHOD_SPLITS
}
aggregate_root = REPO_ROOT / cfg.run.output_dir / f"{BASE_BENCHMARK_VERSION}_aggregate"
summary_dir = aggregate_root / "summary"
figure_dir = aggregate_root / "figures"
summary_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "config": str(CONFIG_PATH.relative_to(REPO_ROOT)),
    "method_splits": list(METHOD_SPLITS),
    "scenario_count": len(scenario_ids),
    "seed_count": len(seed_ids),
    "expected_replicates_per_method": expected_replicates_per_method,
    "allow_partial": ALLOW_PARTIAL,
    "aggregate_root": str(aggregate_root.relative_to(REPO_ROOT)),
}, indent=2))

In [ ]:
def load_manifest(path: Path) -> dict:
    try:
        return json.loads(path.read_text())
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def is_shared_control(variant_id: str) -> bool:
    variant_id = str(variant_id)
    return (
        variant_id in {"clean", "raw", "artifact_oracle"}
        or variant_id.startswith(("pca/", "random_subspace/", "causal_lowpass/"))
    )


def aggregate_variant_id(method_split: str, variant_id: str) -> str:
    variant_id = str(variant_id)
    if variant_id == "oracle_selection":
        return f"{method_split}/oracle_selection"
    return variant_id


def variant_method(method_split: str, variant_id: str) -> str:
    variant_id = str(variant_id)
    if is_shared_control(variant_id):
        return "control"
    if variant_id == "oracle_selection":
        return method_split
    for method in BSS_METHODS:
        if variant_id.startswith(f"{method}/"):
            return method
    return "other"


def assert_no_method_leakage(method_split: str, table: pd.DataFrame) -> None:
    if table.empty or method_split == "fastica" or "variant_id" not in table.columns:
        return
    variants = table["variant_id"].astype(str)
    leaked = variants.str.startswith(("fastica/", "jade_fastica_fallback/"))
    if leaked.any():
        examples = sorted(variants[leaked].unique())[:10]
        raise RuntimeError(
            f"{method_split} split contains FastICA leakage variants: {examples}"
        )


def analysis_dedupe_keys(family: str, table: pd.DataFrame) -> list[str]:
    keys = ["analysis_source", "scenario", "seed", "aggregate_variant_id"]
    if family == "graph" and "estimator" in table.columns:
        keys.append("estimator")
    return keys


def write_csv(table: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False)
    return path

In [ ]:
manifest_rows = []
validation_rows = []

for method, root in split_roots.items():
    if root.exists():
        report = validate_benchmark(root)
        validation_rows.append({
            "method_split": method,
            "root": str(root.relative_to(REPO_ROOT)),
            "checked": report.checked,
            "ok": report.ok,
            "error_count": len(report.errors),
            "errors": "\n".join(report.errors[:20]),
        })
    else:
        validation_rows.append({
            "method_split": method,
            "root": str(root.relative_to(REPO_ROOT)),
            "checked": 0,
            "ok": False,
            "error_count": 1,
            "errors": "missing split root",
        })

    for scenario_id in scenario_ids:
        for seed in seed_ids:
            replicate_dir = root / scenario_id / f"seed_{seed}"
            manifest_path = replicate_dir / "manifest.json"
            manifest = load_manifest(manifest_path)
            manifest_rows.append({
                "method_split": method,
                "scenario": scenario_id,
                "seed": seed,
                "replicate": str(replicate_dir.relative_to(REPO_ROOT)),
                "manifest_exists": manifest_path.exists(),
                "complete": bool(manifest.get("complete")),
                "failure_count": len(manifest.get("failures", [])),
                "n_variants": manifest.get("n_variants"),
                "variant_implementation_version": manifest.get("variant_implementation_version"),
                "git_revision": manifest.get("git_revision"),
            })

manifest_inventory = pd.DataFrame(manifest_rows)
validation_table = pd.DataFrame(validation_rows)
write_csv(manifest_inventory, summary_dir / "manifest_inventory.csv")
write_csv(validation_table, summary_dir / "validation_report.csv")

completion_summary = manifest_inventory.groupby("method_split", as_index=False).agg(
    expected=("manifest_exists", "size"),
    manifests=("manifest_exists", "sum"),
    complete=("complete", "sum"),
    failures=("failure_count", "sum"),
)
completion_summary["missing"] = completion_summary["expected"] - completion_summary["manifests"]
write_csv(completion_summary, summary_dir / "completion_summary.csv")
display(completion_summary)
display(validation_table[["method_split", "checked", "ok", "error_count"]])

bad_validation = validation_table.loc[~validation_table["ok"]]
incomplete = manifest_inventory.loc[
    (~manifest_inventory["manifest_exists"])
    | (~manifest_inventory["complete"])
    | (manifest_inventory["failure_count"] > 0)
]

if not bad_validation.empty:
    raise RuntimeError(
        "At least one split failed artifact validation. Inspect summary/validation_report.csv."
    )
if not ALLOW_PARTIAL and not incomplete.empty:
    raise RuntimeError(
        "Aggregate blocked until every split replicate is complete. "
        "Inspect summary/completion_summary.csv or set ALLOW_PARTIAL=True for preview only."
    )

In [ ]:
metric_tables: dict[str, pd.DataFrame] = {}
metric_families = ("trace", "behavior", "state", "graph")

for family in metric_families:
    frames = []
    for method, root in split_roots.items():
        table = collect_metric_table(root, family)
        if table.empty:
            continue
        assert_no_method_leakage(method, table)
        table = table.copy()
        table.insert(0, "method_split", method)
        table["aggregate_variant_id"] = table["variant_id"].map(
            lambda variant_id: aggregate_variant_id(method, variant_id)
        )
        table["variant_method"] = table["variant_id"].map(
            lambda variant_id: variant_method(method, variant_id)
        )
        table["analysis_source"] = np.where(
            table["variant_method"].eq("control"),
            "shared_control",
            table["method_split"],
        )
        frames.append(table)
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    metric_tables[family] = combined
    write_csv(combined, summary_dir / f"{family}_all_split_rows.csv")
    print(f"{family}: {len(combined):,} split row(s)")

metric_tables.keys()

In [ ]:
analysis_tables: dict[str, pd.DataFrame] = {}

for family, table in metric_tables.items():
    if table.empty:
        analysis_tables[family] = table
        continue
    dedupe_keys = analysis_dedupe_keys(family, table)
    analysis = table.sort_values(["method_split", "scenario", "seed"]).drop_duplicates(
        subset=dedupe_keys,
        keep="first",
    ).copy()
    control_rows = analysis["variant_method"].eq("control")
    analysis.loc[control_rows, "method_split"] = "shared"
    analysis_tables[family] = analysis
    write_csv(analysis, summary_dir / f"{family}_analysis.csv")
    print(
        f"{family}: {len(analysis):,} analysis row(s) "
        f"after removing duplicated shared controls"
    )

In [ ]:
graph = analysis_tables.get("graph", pd.DataFrame())
if graph.empty:
    raise RuntimeError("No graph metrics available after aggregation.")

main_graph = graph[graph["estimator"].isin(MAIN_GRAPH_ESTIMATORS)].copy()
write_csv(main_graph, summary_dir / "graph_analysis_main_estimators.csv")

graph_summary = main_graph.groupby(
    ["scenario", "estimator", "aggregate_variant_id", "variant_method"],
    as_index=False,
).agg(
    f1_median=("f1", "median"),
    f1_q25=("f1", lambda s: s.quantile(0.25)),
    f1_q75=("f1", lambda s: s.quantile(0.75)),
    f1_mean=("f1", "mean"),
    f1_std=("f1", "std"),
    shd_median=("shd", "median"),
    precision_median=("precision", "median"),
    recall_median=("recall", "median"),
    n=("f1", "size"),
)
write_csv(graph_summary, summary_dir / "graph_recovery_summary.csv")

raw = main_graph.loc[
    main_graph["aggregate_variant_id"].eq("raw"),
    ["scenario", "seed", "estimator", "f1", "shd"],
].rename(columns={"f1": "f1_raw", "shd": "shd_raw"})
graph_paired = main_graph.merge(raw, on=["scenario", "seed", "estimator"], how="left")
graph_paired["f1_minus_raw"] = graph_paired["f1"] - graph_paired["f1_raw"]
graph_paired["shd_minus_raw"] = graph_paired["shd"] - graph_paired["shd_raw"]
write_csv(graph_paired, summary_dir / "graph_paired_vs_raw.csv")

bss_graph = graph_paired[graph_paired["variant_method"].isin(BSS_METHODS)].copy()
bss_summary = bss_graph.groupby(
    ["variant_method", "scenario", "estimator", "aggregate_variant_id"],
    as_index=False,
).agg(
    f1_median=("f1", "median"),
    f1_minus_raw_median=("f1_minus_raw", "median"),
    shd_median=("shd", "median"),
    shd_minus_raw_median=("shd_minus_raw", "median"),
    n=("f1", "size"),
)
write_csv(bss_summary, summary_dir / "bss_variant_graph_summary.csv")

if bss_summary.empty:
    best_bss = bss_summary
else:
    idx = bss_summary.groupby(["variant_method", "scenario", "estimator"])["f1_median"].idxmax()
    best_bss = bss_summary.loc[idx].sort_values(["scenario", "estimator", "variant_method"])
write_csv(best_bss, summary_dir / "best_bss_variant_by_scenario_estimator.csv")
display(best_bss.head(30))

In [ ]:
def median_by_variant(table: pd.DataFrame, metrics: dict[str, str]) -> pd.DataFrame:
    if table.empty:
        return pd.DataFrame(columns=["scenario", "aggregate_variant_id", "variant_method"])
    available = {out: col for out, col in metrics.items() if col in table.columns}
    if not available:
        return table[["scenario", "aggregate_variant_id", "variant_method"]].drop_duplicates()
    return table.groupby(["scenario", "aggregate_variant_id", "variant_method"], as_index=False).agg(
        **{out: (col, "median") for out, col in available.items()}
    )


trace_med = median_by_variant(analysis_tables.get("trace", pd.DataFrame()), {
    "trace_pearson_global": "trace_pearson_global",
    "trace_nrmse": "trace_nrmse",
    "artifact_residual_corr": "artifact_residual_corr",
    "effective_rank": "effective_rank",
})
behavior_med = median_by_variant(analysis_tables.get("behavior", pd.DataFrame()), {
    "behavior_pearson": "behavior_pearson",
    "bout_balanced_accuracy": "bout_balanced_accuracy",
})
state_med = median_by_variant(analysis_tables.get("state", pd.DataFrame()), {
    "persistence_improvement": "persistence_improvement",
    "forward_reverse_gap": "forward_reverse_gap",
    "residual_artifact_assoc": "residual_artifact_assoc",
})
graph_med = main_graph.groupby(["scenario", "aggregate_variant_id", "variant_method"], as_index=False).agg(
    graph_f1=("f1", "median"),
    graph_shd=("shd", "median"),
)

tradeoff = trace_med.merge(behavior_med, on=["scenario", "aggregate_variant_id", "variant_method"], how="outer")
tradeoff = tradeoff.merge(state_med, on=["scenario", "aggregate_variant_id", "variant_method"], how="outer")
tradeoff = tradeoff.merge(graph_med, on=["scenario", "aggregate_variant_id", "variant_method"], how="outer")
write_csv(tradeoff, summary_dir / "trace_behavior_state_graph_tradeoff.csv")
tradeoff.sort_values(["scenario", "graph_f1", "trace_pearson_global"], ascending=[True, False, False]).head(30)

In [ ]:
figures = []

fig, ax = plt.subplots(figsize=(8, 3.8))
completion_plot = completion_summary.set_index("method_split")[["complete", "missing"]]
completion_plot.plot(kind="bar", stacked=True, ax=ax, color=["#4c78a8", "#e45756"])
ax.set_ylabel("Replicates")
ax.set_xlabel("Method split")
ax.set_title("Simulation split completion")
ax.legend(loc="upper right")
completion_png = figure_dir / "simulation_split_completion.png"
completion_pdf = figure_dir / "simulation_split_completion.pdf"
fig.savefig(completion_png, dpi=200, bbox_inches="tight")
fig.savefig(completion_pdf, bbox_inches="tight")
figures.append((completion_png, "Completion status for each method split."))
if SHOW_FIGURES:
    plt.show()
else:
    plt.close(fig)

if not best_bss.empty:
    g = sns.catplot(
        data=best_bss,
        kind="bar",
        x="variant_method",
        y="f1_median",
        hue="estimator",
        col="scenario",
        col_wrap=4,
        order=list(METHOD_SPLITS),
        height=3.2,
        aspect=1.05,
        errorbar=None,
        sharey=True,
    )
    g.set_axis_labels("BSS method", "Best median graph F1")
    g.set_titles("{col_name}")
    g.fig.suptitle("Best BSS variant per scenario and estimator", y=1.04)
    out_png = figure_dir / "best_bss_graph_f1.png"
    out_pdf = figure_dir / "best_bss_graph_f1.pdf"
    g.fig.savefig(out_png, dpi=200, bbox_inches="tight")
    g.fig.savefig(out_pdf, bbox_inches="tight")
    figures.append((out_png, "Best median graph F1 for each BSS method, scenario, and estimator."))
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(g.fig)

    g = sns.catplot(
        data=best_bss,
        kind="bar",
        x="variant_method",
        y="f1_minus_raw_median",
        hue="estimator",
        col="scenario",
        col_wrap=4,
        order=list(METHOD_SPLITS),
        height=3.2,
        aspect=1.05,
        errorbar=None,
        sharey=True,
    )
    g.set_axis_labels("BSS method", "Best median F1 minus raw")
    g.set_titles("{col_name}")
    for ax in g.axes.flat:
        ax.axhline(0.0, color="black", linewidth=1)
    g.fig.suptitle("Best BSS graph-recovery effect relative to raw", y=1.04)
    out_png = figure_dir / "best_bss_f1_minus_raw.png"
    out_pdf = figure_dir / "best_bss_f1_minus_raw.pdf"
    g.fig.savefig(out_png, dpi=200, bbox_inches="tight")
    g.fig.savefig(out_pdf, bbox_inches="tight")
    figures.append((out_png, "Best BSS graph F1 effect relative to raw traces."))
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(g.fig)

if {"trace_pearson_global", "graph_f1"}.issubset(tradeoff.columns):
    plot_data = tradeoff[tradeoff["variant_method"].isin(["control", *METHOD_SPLITS])].copy()
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(
        data=plot_data,
        x="trace_pearson_global",
        y="graph_f1",
        hue="variant_method",
        style="scenario",
        s=80,
        ax=ax,
    )
    ax.set_xlabel("Trace recovery, median Pearson vs clean")
    ax.set_ylabel("Graph recovery, median F1")
    ax.set_title("Trace recovery versus graph recovery")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    out_png = figure_dir / "trace_vs_graph_recovery.png"
    out_pdf = figure_dir / "trace_vs_graph_recovery.pdf"
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    figures.append((out_png, "Trace preservation versus causal graph recovery trade-off."))
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(fig)

figures

In [ ]:
catalog_lines = [
    "# Simulation Aggregate Figure Catalog",
    "",
    f"Source benchmark prefix: `{BASE_BENCHMARK_VERSION}`",
    f"Method splits: {', '.join(METHOD_SPLITS)}",
    f"Expected replicates per method: {expected_replicates_per_method}",
    "",
]
for path, purpose in figures:
    rel = path.relative_to(REPO_ROOT)
    catalog_lines.extend([
        f"## `{rel}`",
        "",
        f"Purpose: {purpose}",
        "Data source: aggregate CSVs in `outputs/simulation/core_1000_aggregate/summary/`.",
        "Caveat: final manuscript claims require all method splits to be complete and validated.",
        "",
    ])

(summary_dir / "figure_catalog.md").write_text("\n".join(catalog_lines))

generated_csvs = sorted(path.relative_to(REPO_ROOT) for path in summary_dir.glob("*.csv"))
generated_figures = sorted(path.relative_to(REPO_ROOT) for path in figure_dir.glob("*.png"))
print("Summary CSVs:")
for path in generated_csvs:
    print(f"- {path}")
print("\nFigures:")
for path in generated_figures:
    print(f"- {path}")